In [ ]:
import torch 
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# 【显卡点火核心】：检测你的电脑是否有 N 卡，有就直接接管！
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🔥 当前核动力引擎: {device}")

df = pd.read_csv(r'D:\PycharmWork\pythonProject\IEEE33_50000_Safe_Final.csv')
raw_data = df.values
m = raw_data.shape[0]
print(raw_data.shape)
data_3d = raw_data.reshape(m , 33, 4)
print(data_3d.shape)
X_3d = data_3d[:,:,0:2]
Y_3d = data_3d[:,:,2:4]
print(X_3d.shape)
print(Y_3d.shape)
X_3d = X_3d.reshape(m , 66)
Y_3d = Y_3d.reshape(m , 66)
print(X_3d.shape)
print(Y_3d.shape)
ssl = StandardScaler()
X_3d = ssl.fit_transform(X_3d)

# 【改动 1】：把 5 万发洗净的弹药，直接装填进显卡的显存里！
X = torch.tensor(X_3d , dtype = torch.float32).to(device)
Y = torch.tensor(Y_3d , dtype = torch.float32).to(device)
loss_list = []

# 【改动 2】：手工捏造的权重 W 和偏置 b，也必须直接在显卡上铸造！
W = (torch.randn((66,66) , dtype = torch.float32).to(device)) * 0.01
b = torch.zeros((1 , 66) , dtype = torch.float32).to(device)
learning_rate = 0.01
m = X.shape[0]
for epoch in range(50000):
    y_pred = torch.matmul(X, W) + b
    error = y_pred - Y
    loss = torch.mean((y_pred - Y)**2)
    loss_list.append(loss.item()) 
    
    dW = (2 / m) *torch.matmul(X.T, error)
    db = (2 / m) *torch.sum(error , dim = 0)
    W = W - learning_rate * dW
    b = b - learning_rate * db
# 设置 300 DPI 高清与学术字体风格
plt.rcParams['figure.dpi'] = 300
plt.rcParams['font.sans-serif'] = ['SimHei'] # 支持中文显示
plt.rcParams['axes.unicode_minus'] = False

plt.figure(figsize=(8, 5))
plt.plot(loss_list, color='red', linewidth=1.5)
plt.title("手工微积分引擎：IEEE 33节点电压预测误差收敛图", fontweight='bold')
plt.xlabel("训练轮次 (Epochs)")
plt.ylabel("均方误差 (MSE Loss)")
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
import torch 
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader,random_split
import torch.nn as nn
class MyDataset(Dataset):
    def __init__(self , features , labels):
        self.features = features
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, index):
        return self.features[index], self.labels[index]
# 【显卡点火核心】：检测你的电脑是否有 N 卡，有就直接接管！
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🔥 当前核动力引擎: {device}")

df = pd.read_csv(r'D:\PycharmWork\pythonProject\IEEE33_50000_Safe_Final.csv')
raw_data = df.values
m = raw_data.shape[0]
print(raw_data.shape)
data_3d = raw_data.reshape(m , 33, 4)
print(data_3d.shape)
X_3d = data_3d[:,:,0:2]
Y_3d = data_3d[:,:,2:4]
print(X_3d.shape)
print(Y_3d.shape)
X_3d = X_3d.reshape(m , 66)
Y_3d = Y_3d.reshape(m , 66)
print(X_3d.shape)
print(Y_3d.shape)
ssl = StandardScaler()
X_3d = ssl.fit_transform(X_3d)
#现在开始投毒
noise = np.random.normal(loc = 0. , scale = 1. , size = X_3d.shape )
X_3d_dirty = X_3d + noise
# 【改动 1】：把 5 万发洗净的弹药，直接装填进显卡的显存里！
X = torch.tensor(X_3d_dirty , dtype = torch.float32).to(device)
Y = torch.tensor(Y_3d , dtype = torch.float32).to(device)
loss_list = []

# # 【改动 2】：手工捏造的权重 W 和偏置 b，也必须直接在显卡上铸造！
# W = (torch.randn((66,66) , dtype = torch.float32).to(device)) * 0.01
# b = torch.zeros((1 , 66) , dtype = torch.float32).to(device)
grid_data = MyDataset(X, Y)
train_loader = DataLoader(dataset=grid_data, batch_size=1000, shuffle=True)
for batch_X , batch_Y in train_loader:
    print(batch_X.shape)
    print(batch_Y.shape)
    break
total_count = len(grid_data)
train_count = int(total_count * 0.8)
val_count = int(total_count * 0.1)
test_count = total_count - train_count - val_count
train_db , val_db , test_db = random_split(grid_data , [train_count , val_count , test_count])
train_loader = DataLoader(dataset=train_db, batch_size=1000, shuffle=True)
val_loader = DataLoader(dataset=val_db, batch_size=1000, shuffle=False)
test_loader = DataLoader(dataset=test_db, batch_size=1000, shuffle=False)
#model更新 加入dropout 随机击毙神经元
class PowerMLP(nn.Module):
    def __init__(self):
        super(PowerMLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features = 66 , out_features = 128),
            nn.ReLU(),
            nn.Dropout(p = 0.2),
            nn.Linear(128,64),
            nn.ReLU(),
            nn.Dropout(p = 0.2),
            nn.Linear(64,66),
        )
    def forward(self , x):
        return self.net(x)
#手写PINN
class PowerPINNLoss(nn.Module):
    def __init__(self):
        super(PowerPINNLoss, self).__init__()
        self.loss = nn.MSELoss()
    def forward(self , pred, target):
        data_loss = self.loss(pred, target)
        pred_3d = pred.reshape(-1 , 33 ,2)
        Vm_pred = pred_3d[: , : , 0]
        penalty_low = torch.mean(torch.relu(0.9 - Vm_pred))
        penalty_high = torch.mean(torch.relu(Vm_pred - 1.1))
        return data_loss + 10 * penalty_low + 10 * penalty_high
model = PowerMLP().to(device)
print(model)
# criterion = nn.MSELoss() 现在开始用powerpinnloss
criterion = PowerPINNLoss().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
epochs = 150
train_loss_list = []
val_loss_list = []
for epoch in range(epochs):
    model.train()
    total_train_loss = 0
    for batch_X, batch_Y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_Y)
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()
    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for v_X , v_Y in val_loader:
            v_outputs = model(v_X)
            loss = criterion(v_outputs, v_Y)
            total_val_loss += loss.item()
    avg_train = total_train_loss / len(train_loader)
    avg_val = total_val_loss / len(val_loader)
    
    train_loss_list.append(avg_train)
    val_loss_list.append(avg_val)
    if (epoch + 1) % 10 == 0:
        print(f"   ➤ 第 {epoch+1:3d} 圈 | 训练误差: {avg_train:.6f} | 验证误差: {avg_val:.6f}")
        plt.figure(figsize=(8, 5))
plt.rcParams['figure.dpi'] = 300
plt.rcParams['font.sans-serif'] = ['SimHei'] # 支持中文显示
plt.rcParams['axes.unicode_minus'] = False

plt.figure(figsize=(8, 5))
# 画出两条线，对比看是否过拟合
plt.plot(train_loss_list, color='red', linewidth=1.5, label='训练误差 (Train)')
plt.plot(val_loss_list, color='blue', linestyle='--', linewidth=1.5, label='验证误差 (Val)')

# 标题升级：我们已经是多层感知机了！
plt.title("深度学习 MLP 引擎：IEEE 33节点电压预测收敛图", fontweight='bold')
plt.xlabel("训练轮次 (Epochs)")
plt.ylabel("均方误差 (MSE Loss)")
plt.legend() # 显示图例
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()
#开始大调查
model.eval()
low_violations_count = 0
high_violations_count = 0
total_bus = 0
with torch.no_grad():
    for batch_X, batch_Y in test_loader:
        t_output = model(batch_X)
        pred_3d = t_output.reshape(-1 , 33 ,2)
        Vm_pred = pred_3d[: , : , 0]
        low_violations_count += torch.sum(Vm_pred<0.9).item()
        high_violations_count += torch.sum(Vm_pred>1.1).item()
        total_bus += Vm_pred.numel()
print(f'总共测试节点{total_bus} ， 有多少超出 低：{low_violations_count} , 高：{high_violations_count}')

In [ ]:
import torch 
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader,random_split
import torch.nn as nn
class MyDataset(Dataset):
    def __init__(self , features , labels):
        self.features = features
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, index):
        return self.features[index], self.labels[index]
# 【显卡点火核心】：检测你的电脑是否有 N 卡，有就直接接管！
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🔥 当前核动力引擎: {device}")

df = pd.read_csv(r'D:\PycharmWork\pythonProject\IEEE33_50000_Safe_Final.csv')
raw_data = df.values
m = raw_data.shape[0]
print(raw_data.shape)
data_3d = raw_data.reshape(m , 33, 4)
print(data_3d.shape)
X_3d = data_3d[:,:,0:2]
Y_3d = data_3d[:,:,2:4]
print(X_3d.shape)
print(Y_3d.shape)
X_3d = X_3d.reshape(m , 66)
Y_3d = Y_3d.reshape(m , 66)
print(X_3d.shape)
print(Y_3d.shape)
ssl = StandardScaler()
X_3d = ssl.fit_transform(X_3d)
#现在开始投毒
noise = np.random.normal(loc = 0. , scale = 1. , size = X_3d.shape )
X_3d_dirty= X_3d + noise
# 【改动 1】：把 5 万发洗净的弹药，直接装填进显卡的显存里！
X = torch.tensor(X_3d_dirty , dtype = torch.float32).to(device)
Y = torch.tensor(Y_3d , dtype = torch.float32).to(device)
loss_list = []

# # 【改动 2】：手工捏造的权重 W 和偏置 b，也必须直接在显卡上铸造！
# W = (torch.randn((66,66) , dtype = torch.float32).to(device)) * 0.01
# b = torch.zeros((1 , 66) , dtype = torch.float32).to(device)
grid_data = MyDataset(X, Y)
train_loader = DataLoader(dataset=grid_data, batch_size=1000, shuffle=True)
for batch_X , batch_Y in train_loader:
    print(batch_X.shape)
    print(batch_Y.shape)
    break
total_count = len(grid_data)
train_count = int(total_count * 0.8)
val_count = int(total_count * 0.1)
test_count = total_count - train_count - val_count
train_db , val_db , test_db = random_split(grid_data , [train_count , val_count , test_count])
train_loader = DataLoader(dataset=train_db, batch_size=1000, shuffle=True)
val_loader = DataLoader(dataset=val_db, batch_size=1000, shuffle=False)
test_loader = DataLoader(dataset=test_db, batch_size=1000, shuffle=False)
#model更新 加入dropout 随机击毙神经元
class PowerMLP(nn.Module):
    def __init__(self):
        super(PowerMLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features = 66 , out_features = 128),
            nn.ReLU(),
            nn.Dropout(p = 0.2),
            nn.Linear(128,64),
            nn.ReLU(),
            nn.Dropout(p = 0.2),
            nn.Linear(64,66),
        )
    def forward(self , x):
        return self.net(x)
#手写PINN
class PowerPINNLoss(nn.Module):
    def __init__(self):
        super(PowerPINNLoss, self).__init__()
        self.loss = nn.MSELoss()
    def forward(self , pred, target):
        data_loss = self.loss(pred, target)
        pred_3d = pred.reshape(-1 , 33 ,2)
        Vm_pred = pred_3d[: , : , 0]
        penalty_low = torch.mean(torch.relu(0.9 - Vm_pred))
        penalty_high = torch.mean(torch.relu(Vm_pred - 1.1))
        return data_loss + 10 * penalty_low + 10 * penalty_high
model = PowerMLP().to(device)
print(model)
#对比PINN
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
epochs = 150
train_loss_list = []
val_loss_list = []
for epoch in range(epochs):
    model.train()
    total_train_loss = 0
    for batch_X, batch_Y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_Y)
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()
    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for v_X , v_Y in val_loader:
            v_outputs = model(v_X)
            loss = criterion(v_outputs, v_Y)
            total_val_loss += loss.item()
    avg_train = total_train_loss / len(train_loader)
    avg_val = total_val_loss / len(val_loader)
    
    train_loss_list.append(avg_train)
    val_loss_list.append(avg_val)
    if (epoch + 1) % 10 == 0:
        print(f"   ➤ 第 {epoch+1:3d} 圈 | 训练误差: {avg_train:.6f} | 验证误差: {avg_val:.6f}")
        plt.figure(figsize=(8, 5))
plt.rcParams['figure.dpi'] = 300
plt.rcParams['font.sans-serif'] = ['SimHei'] # 支持中文显示
plt.rcParams['axes.unicode_minus'] = False

plt.figure(figsize=(8, 5))
# 画出两条线，对比看是否过拟合
plt.plot(train_loss_list, color='red', linewidth=1.5, label='训练误差 (Train)')
plt.plot(val_loss_list, color='blue', linestyle='--', linewidth=1.5, label='验证误差 (Val)')

# 标题升级：我们已经是多层感知机了！
plt.title("深度学习 MLP 引擎：IEEE 33节点电压预测收敛图", fontweight='bold')
plt.xlabel("训练轮次 (Epochs)")
plt.ylabel("均方误差 (MSE Loss)")
plt.legend() # 显示图例
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()
#开始大调查
model.eval()
low_violations_count = 0
high_violations_count = 0
total_bus = 0
with torch.no_grad():
    for batch_X, batch_Y in test_loader:
        t_output = model(batch_X)
        pred_3d = t_output.reshape(-1 , 33 , 2)
        Vm_pred = pred_3d[: , : ,0]
        low_violations_count += torch.sum(Vm_pred<0.9).item()
        high_violations_count += torch.sum(Vm_pred>1.1).item()
        total_bus += Vm_pred.numel()
print(f'总共测试节点{total_bus} ， 有多少超出 低：{low_violations_count} , 高：{high_violations_count}')

In [ ]:
import numpy as np

# IEEE 33 节点系统的支路数据 (共 32 条支路)
# 格式: [起点(From), 终点(To), 电阻 R (p.u.), 电抗 X (p.u.)]
branch_data = np.array([
    [1, 2, 0.00057, 0.00029], [2, 3, 0.00307, 0.00156], [3, 4, 0.00228, 0.00117],
    [4, 5, 0.00237, 0.00121], [5, 6, 0.00511, 0.00441], [6, 7, 0.00116, 0.00336],
    [7, 8, 0.00443, 0.00146], [8, 9, 0.00642, 0.00461], [9, 10, 0.00651, 0.00461],
    [10, 11, 0.00122, 0.00040], [11, 12, 0.00233, 0.00074], [12, 13, 0.00915, 0.00720],
    [13, 14, 0.00337, 0.00444], [14, 15, 0.00368, 0.00328], [15, 16, 0.00465, 0.00340],
    [16, 17, 0.00804, 0.01073], [17, 18, 0.00456, 0.00358], [2, 19, 0.00102, 0.00097],
    [19, 20, 0.00938, 0.00845], [20, 21, 0.00255, 0.00298], [21, 22, 0.00442, 0.00584],
    [3, 23, 0.00281, 0.00192], [23, 24, 0.00560, 0.00442], [24, 25, 0.00559, 0.00437],
    [6, 26, 0.00126, 0.00064], [26, 27, 0.00177, 0.00090], [27, 28, 0.00660, 0.00582],
    [28, 29, 0.00501, 0.00437], [29, 30, 0.00316, 0.00161], [30, 31, 0.00608, 0.00600],
    [31, 32, 0.00193, 0.00225], [32, 33, 0.00212, 0.00330]
])
R = branch_data[: , 2]
X = branch_data[: , 3]
num_nodes =33
y_bus = np.zeros((num_nodes , num_nodes) , dtype = complex)
print(y_bus)
for i in range(len(branch_data)):
    f = branch_data[i , 0]-1
    t = branch_data[i , 1]-1
    r = branch_data[i , 2]
    x = branch_data[i , 3]
    #开始算电导
    z_line = r + 1j * x
    y_line = 1 / z_line
    #主对角线 就是自导
    y_bus[f,f] += y_line
    y_bus[t,t] += y_line
    #非对角线 就是互导
    y_bus[f , t] = -y_line
    y_bus[t , f] = -y_line